# 7. Chuẩn hóa, hợp nhất references và khử trùng

## 7.1 Chuẩn hóa nội dung

Normalizer bỏ các wrapper trình bày trong tập đã hỗ trợ, giữ nội dung bên trong và chuẩn
hóa whitespace/Unicode NFC. Inline math đưa về `$…$`, block math về môi trường equation.
Các đoạn text trong toán và literal được bảo vệ để tránh xóa khoảng trắng có ý nghĩa.
Những lệnh chưa biết không bị xóa mù, nên một số cú pháp vẫn còn trong content chuẩn hóa.

Citation key được đổi bằng mapping của từng version, kể cả cite nằm trong toán.
Chuẩn hóa là thao tác trên chuỗi TeX, không phải chứng minh hai biểu thức toán tương đương.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
from parser.normalize import normalize
examples = [r"\textbf{Shared} text with \(x + y\).", r"\centering Shared text with $x+y$."]
display(pd.DataFrame([{'illustrative_raw':text,'normalized':normalize(text)} for text in examples]))

## 7.2 Trích và hợp nhất BibTeX

Entries được lấy từ bibliography thuộc compilation path và các bibitem trong TeX.
Bibitem được phân tích thành trường có thể xác định; không tự điền DOI/year khi không có
bằng chứng. Parser BibTeX giữ `%` trong giá trị hoặc URL, phân biệt với comment bên ngoài.

Hợp nhất ưu tiên định danh mạnh có thể nhận diện, sau đó dùng chữ ký title/author khi
không có mâu thuẫn year/DOI. Nếu chưa có nhận diện đủ tin cậy, entry không bị gộp chỉ vì
key giống nhau. Field mới bổ sung vào bản hợp nhất; giá trị mâu thuẫn được ghi riêng,
không tùy tiện ghi đè để làm mất thông tin.

Ví dụ sau dùng entries tổng hợp. Version 1 có year, version 2 thêm DOI và dùng key khác.
Kết quả cho thấy field union và mapping được tạo bởi code thực thi.

In [ ]:
from parser.bibliography import parse_bibtex, unify_entries
entries = parse_bibtex('@article{first,title={Sample reference},author={A. Example},year={2020}}',1)
entries += parse_bibtex('@article{second,title={Sample reference},author={A. Example},doi={10.0000/demo}}',2)
rows, mappings, conflicts = unify_entries('demo',entries)
display(pd.DataFrame(rows).drop(columns=['arxiv_id']))
print('Citation mapping:',mappings)
print('Normalized cite, version 2:',normalize(r'We cite \cite{second}.',mappings[2]))
print('Conflicts:',conflicts)

## 7.3 Content store dùng chung giữa các version

Lookup dùng cặp `(kind, normalized_content)`. Nội dung chuẩn hóa giống hệt và cùng kind
được lưu một lần; ID được tạo ổn định từ nội dung và paper key. Các occurrence vẫn giữ
node gốc, version và vị trí. Bảng hierarchy lưu cạnh riêng cho từng version, nên một
sentence dùng chung có thể nằm dưới hai heading khác nhau.

Không gộp hai publication với nhau. Không dùng fuzzy matching, embedding hoặc độ tương
đồng xấp xỉ để quyết định content dedup của milestone này. Mọi self-edge phát sinh do
gộp cấu trúc phải bị phát hiện, không được lặng lẽ xóa để validator đi qua.

Ví dụ tổng hợp dưới đây có một câu dùng chung nhưng heading đổi giữa hai version.

In [ ]:
from parser.hierarchy import parse_hierarchy
from parser.deduplicate import deduplicate
first = r"\documentclass{article}\title{Demo}\begin{document}\section{Background}Shared sentence.\end{document}"
second = r"\documentclass{article}\title{Demo}\begin{document}\section{Method}Shared sentence. New sentence.\end{document}"
trees={1:parse_hierarchy(first),2:parse_hierarchy(second)}
elements,edges,occurrences=deduplicate('demo',trees,{})
aliases={row['element_id']:f'E{i:02d}' for i,row in enumerate(elements,1)}
display(pd.DataFrame([{'element':aliases[r['element_id']],'kind':r['kind'],'content':r['content']} for r in elements]))
display(pd.DataFrame([{'version':r['version'],'element':aliases[r['element_id']],'parent':aliases[r['parent_id']]} for r in edges]))
print('Occurrences trước dedup:',sum(len(t.nodes) for t in trees.values()),'; store sau dedup:',len(elements))

## 7.4 Kết quả và công thức

Với mỗi loại store, tỷ lệ giảm là $(N_{before}-N_{after})/N_{before}$.
Before-counts lấy từ version có đo được; after-counts là số row store của các paper đã
xử lý. Các version chưa parse được không được tự cấp before-counts bằng một số ước lượng.

In [ ]:
dedup=pd.DataFrame(m2['deduplication']).T
dedup['removed']=dedup['before']-dedup['after']; display(dedup)
dedup[['before','after']].plot.bar(title='Counts trước/sau dedup trên phần đã đo')
plt.tight_layout(); plt.show()
print('Version chưa đo before-counts:',m2.get('versions_with_unmeasured_counts'))
print('Tổng unresolved citation keys theo version:',m2['unresolved_cites'])
print('Bibliography field conflicts ghi nhận:',m2['bibliography_conflicts'])
display(Markdown(f"Elements giảm **{m2['deduplication']['elements']['reduction_rate']:.2%}**; "
                 f"BibTeX entries giảm **{m2['deduplication']['bibtex']['reduction_rate']:.2%}** trên phần kết quả đã đo."))

## 7.5 Diễn giải

Giảm số dòng cho thấy nội dung giữa các version có thể dùng chung store; nó không tự
chứng minh kết quả đúng ngữ nghĩa hay toàn bộ references đã có. Unresolved keys được
đếm theo version nên cùng một key còn thiếu ở hai version đóng góp hai lần. Field conflict
là số sự kiện mâu thuẫn được ghi nhận, không phải số publication lỗi.

Cần đọc các ca partial, đối chiếu nguồn bibliography và kiểm tra những key cùng tên
trỏ tới nội dung khác nhau. Đó là phần hoàn thiện dữ liệu thực tế, không phải thay đổi
số liệu để giảm unresolved hoặc tăng tỷ lệ success.

## Tài liệu và mã tham chiếu

- [Đặc tả M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/parser/normalize.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/parser/normalize.py).
- Mã triển khai: [src/parser/bibliography.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/parser/bibliography.py).
- Mã triển khai: [src/parser/deduplicate.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/parser/deduplicate.py).
- Mã triển khai: [src/parser/statistics.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/parser/statistics.py).